In [ ]:
# Import required packages
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

In [ ]:
data = pd.read_pickle(INPUT_DATA)

In [ ]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Select features
FEATURES = ['net_sentiment', 'log_volume']

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

In [ ]:
# Define PyTorch Neural Network Model
class NeuralNetRegressor(nn.Module):
    def __init__(self, input_size, hidden_sizes=(16, 8, 4), dropout_rate=0.1):
        super(NeuralNetRegressor, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        # Output layer
        layers.append(nn.Linear(prev_size, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze()


def train_pytorch_model(X_train, y_train, hidden_sizes=(16, 8, 4), 
                        learning_rate=0.001, max_epochs=200, batch_size=1024,
                        early_stopping_patience=10, validation_fraction=0.1,
                        device='cuda'):
    """
    Train a PyTorch neural network model on GPU.
    """
    # Convert to tensors
    X_tensor = torch.FloatTensor(X_train).to(device)
    y_tensor = torch.FloatTensor(y_train.values if hasattr(y_train, 'values') else y_train).to(device)
    
    # Split into training and validation
    n_samples = len(X_tensor)
    n_val = int(n_samples * validation_fraction)
    indices = torch.randperm(n_samples)
    
    train_indices = indices[n_val:]
    val_indices = indices[:n_val]
    
    X_train_t = X_tensor[train_indices]
    y_train_t = y_tensor[train_indices]
    X_val_t = X_tensor[val_indices]
    y_val_t = y_tensor[val_indices]
    
    # Create data loader for training
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Initialize model
    input_size = X_train.shape[1]
    model = NeuralNetRegressor(input_size, hidden_sizes).to(device)
    
    # Loss and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    # Training loop with early stopping
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(max_epochs):
        # Training phase
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t)
            val_loss = criterion(val_outputs, y_val_t).item()
        
        scheduler.step(val_loss)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model


def predict_pytorch(model, X_test, scaler, device='cuda'):
    """
    Generate predictions using a trained PyTorch model.
    """
    model.eval()
    X_scaled = scaler.transform(X_test)
    X_tensor = torch.FloatTensor(X_scaled).to(device)
    
    with torch.no_grad():
        predictions = model(X_tensor).cpu().numpy()
    
    return predictions

# In-Sample Neural Network (GPU)

In [ ]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Standardize features for neural network
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train PyTorch model on GPU
print("Training in-sample model on GPU...")
nn_model = train_pytorch_model(
    X_scaled, y,
    hidden_sizes=(16, 8, 4),
    learning_rate=0.001,
    max_epochs=200,
    batch_size=4096,
    early_stopping_patience=10,
    device=device
)

# Make predictions
nn_model.eval()
X_tensor = torch.FloatTensor(X_scaled).to(device)
with torch.no_grad():
    y_pred = nn_model(X_tensor).cpu().numpy()

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("\nIn-Sample Neural Network Results (GPU)")
print("=" * 50)
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print(f"\nModel Architecture: {(16, 8, 4)}")

# OOS Predictions (Monthly Training, Daily Predictions) - GPU

In [ ]:
# Out-of-sample predictions with MONTHLY TRAINING but DAILY PREDICTIONS
# The model is trained once per month (at month-end) and used to predict all days in the following month

# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW_252 = 252  # One trading year (in days) for rolling window
WINDOW_21 = 21    # One trading month (in days) for rolling window

# Neural network parameters for OOS
NN_HIDDEN_SIZES = (16, 8, 4)
NN_LEARNING_RATE = 0.001
NN_MAX_EPOCHS = 200
NN_BATCH_SIZE = 4096  # Larger batch size for GPU efficiency
NN_EARLY_STOPPING = 10

# Ensure date column is datetime
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])

# Sort by date
model_data = model_data.sort_values('date')

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Create a year-month column for grouping
model_data['year_month'] = model_data['date'].dt.to_period('M')

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")
print(f"Using device: {device}")

In [ ]:
# Initialize storage for predictions
predictions_expanding = []
predictions_rolling_252 = []
predictions_rolling_21 = []

# Loop through OOS months (train once per month)
for month_idx, pred_month in enumerate(oos_months):
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    
    if len(month_dates) == 0:
        continue
    
    # Training cutoff: end of the previous month (first day of pred_month - 1 day)
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    
    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        continue
    last_train_date = train_dates[-1]
    
    # 1. EXPANDING WINDOW: Train on all data up to end of previous month
    train_mask_exp = model_data['date'] <= last_train_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        # Standardize features
        scaler_exp = StandardScaler()
        X_train_exp_scaled = scaler_exp.fit_transform(X_train_exp)
        
        # Train model on GPU
        nn_exp = train_pytorch_model(
            X_train_exp_scaled, y_train_exp,
            hidden_sizes=NN_HIDDEN_SIZES,
            learning_rate=NN_LEARNING_RATE,
            max_epochs=NN_MAX_EPOCHS,
            batch_size=NN_BATCH_SIZE,
            early_stopping_patience=NN_EARLY_STOPPING,
            device=device
        )
        
        # Use this model to predict for all days in the month
        for pred_date in month_dates:
            test_mask = model_data['date'] == pred_date
            X_test = model_data.loc[test_mask, FEATURES]
            
            if len(X_test) == 0:
                continue
            
            test_indices = model_data.index[test_mask]
            y_pred_exp = predict_pytorch(nn_exp, X_test, scaler_exp, device)
            
            for idx, pred in zip(test_indices, y_pred_exp):
                predictions_expanding.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_expanding': pred
                })
    
    # 2. ROLLING 252-DAY WINDOW: Train on last 252 trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx >= WINDOW_252:
        start_date_252 = unique_dates[last_train_date_idx - WINDOW_252 + 1]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] <= last_train_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            # Standardize features
            scaler_252 = StandardScaler()
            X_train_252_scaled = scaler_252.fit_transform(X_train_252)
            
            # Train model on GPU
            nn_252 = train_pytorch_model(
                X_train_252_scaled, y_train_252,
                hidden_sizes=NN_HIDDEN_SIZES,
                learning_rate=NN_LEARNING_RATE,
                max_epochs=NN_MAX_EPOCHS,
                batch_size=NN_BATCH_SIZE,
                early_stopping_patience=NN_EARLY_STOPPING,
                device=device
            )
            
            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]
                
                if len(X_test) == 0:
                    continue
                
                test_indices = model_data.index[test_mask]
                y_pred_252 = predict_pytorch(nn_252, X_test, scaler_252, device)
                
                for idx, pred in zip(test_indices, y_pred_252):
                    predictions_rolling_252.append({
                        'date': pred_date,
                        'index': idx,
                        'pred_rolling_252': pred
                    })
    
    # 3. ROLLING 21-DAY WINDOW: Train on last 21 trading days before the month
    if last_train_date_idx >= WINDOW_21:
        start_date_21 = unique_dates[last_train_date_idx - WINDOW_21 + 1]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] <= last_train_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            # Standardize features
            scaler_21 = StandardScaler()
            X_train_21_scaled = scaler_21.fit_transform(X_train_21)
            
            # Train model on GPU
            nn_21 = train_pytorch_model(
                X_train_21_scaled, y_train_21,
                hidden_sizes=NN_HIDDEN_SIZES,
                learning_rate=NN_LEARNING_RATE,
                max_epochs=NN_MAX_EPOCHS,
                batch_size=NN_BATCH_SIZE,
                early_stopping_patience=NN_EARLY_STOPPING,
                device=device
            )
            
            # Use this model to predict for all days in the month
            for pred_date in month_dates:
                test_mask = model_data['date'] == pred_date
                X_test = model_data.loc[test_mask, FEATURES]
                
                if len(X_test) == 0:
                    continue
                
                test_indices = model_data.index[test_mask]
                y_pred_21 = predict_pytorch(nn_21, X_test, scaler_21, device)
                
                for idx, pred in zip(test_indices, y_pred_21):
                    predictions_rolling_21.append({
                        'date': pred_date,
                        'index': idx,
                        'pred_rolling_21': pred
                    })
    
    # Progress update
    if (month_idx + 1) % 12 == 0:
        print(f"Processed {month_idx + 1}/{len(oos_months)} months ({100 * (month_idx + 1) / len(oos_months):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_expanding):,}")
print(f"Rolling 252-day predictions: {len(predictions_rolling_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_rolling_21):,}")

In [ ]:
# Convert predictions to DataFrames and merge
df_expanding = pd.DataFrame(predictions_expanding)
df_rolling_252 = pd.DataFrame(predictions_rolling_252)
df_rolling_21 = pd.DataFrame(predictions_rolling_21)

# Merge all predictions together
predictions_df = df_expanding.merge(
    df_rolling_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_rolling_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"\nNon-null predictions by window type:")
print(f"  Expanding: {predictions_df['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_df['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_df['pred_rolling_21'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

In [ ]:
# Save predictions to Data directory
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, "predictions_neural_network_monthly_training_gpu.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")

In [ ]:
# Clean up GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")